# Урок 11. Очистка данных: пропуски, дубликаты, типы, выбросы

**Дата:** 03.12.2026 · **60 минут** · 8–9 класс  
**Модуль 3.** Данные (уроки 10–16)  
**Опирается на:** урок 10 (pandas: паспорт, фильтры, новые столбцы), урок 6 (чистка строк), урок 3 (типы).

Главный закон работы с данными: **мусор на входе — мусор на выходе**. Если в таблице половина значений пустая, один ученик записан дважды, а баллы хранятся текстом, любая модель выучит хаос. Сегодня учимся **находить** проблемы в данных и **чинить** их — с объяснением, почему именно так.

## Цели урока

К концу занятия ты сможешь:

1. Находить пропуски (`isna`) и решать, что с ними делать.
2. Находить и удалять дубликаты.
3. Приводить текстовые категории к единому виду.
4. Превращать «текстовые числа» в настоящие (`to_numeric`).
5. Замечать выбросы и вести **журнал решений** по очистке.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Откуда берётся «грязь» |
| 5–15 | Паспорт грязной таблицы: находим проблемы |
| 15–40 | Чиним по порядку: дубликаты → текст → типы → выбросы → пропуски |
| 40–55 | Практика: чистим таблицу магазина |
| 55–60 | Итог и домашнее задание |

## 1. Откуда берётся грязь в данных

| Источник | Пример |
|----------|--------|
| Ручной ввод | забыли заполнить поле, написали «девятый» вместо 9 |
| Разные люди | `Python`, `python`, `python ` — для компьютера три разных значения |
| Сбой при выгрузке | одна строка попала в файл дважды |
| Опечатка | балл 200 при шкале 0–100 |

| Проблема | Аналогия |
|----------|----------|
| Пропуск | пустая клетка в анкете |
| Дубликат | одного ученика записали в журнал дважды |
| Неверный тип | в графе «класс» написано словом |
| Выброс | рост 3 метра в школьной ведомости |

Правило инженера: **сначала диагностика, потом лечение**. И каждое решение записываем.

## 2. Грязный журнал кружков

Это выгрузка из онлайн-формы, которую заполняли сами ученики. Пустые значения записаны как `None` — в pandas они станут **пропусками**.

In [ ]:
import pandas as pd

raw = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 3, 11, 12, 13, 14],
    "имя": ["Аня", "Боря", "Вика", "Глеб", "Даша", "Егор", "Женя", "Зоя", "Илья", "Кира",
            "Вика", "Лёша", "Мила", "Никита", "Оля"],
    "кружок": ["Python", "роботы", "Python", "Игры", None, "python ", "Игры", "Роботы", "Роботы", "Python",
               "Python", "игры", "Роботы", None, "Python"],
    "класс": ["8", "9", "8", "девятый", "8", "8", "9", "9", "8", "9", "8", "9", "", "9", "8"],
    "часы": [8, 3, 10, None, 7, 2, 9, 6, 4, 11, 10, 8, None, 12, 3],
    "балл": [85, 42, 91, 58, 74, 35, 88, 200, 51, 96, 91, 82, 63, 40, 71],
})
raw.to_csv("dirty_journal.csv", index=False)

df = pd.read_csv("dirty_journal.csv")
df

**Найди глазами** до того, как читать дальше:

1. Кто записан дважды?
2. Где пустые ячейки?
3. Какие проблемы в столбце `кружок`?
4. Что не так со столбцом `класс`?
5. Какой балл подозрителен?

## 3. Паспорт до очистки

Всегда сохраняй картину «до»: иначе не докажешь, что стало лучше.

In [ ]:
print("Размер:", df.shape)
df.info()

Что уже видно:
- в столбцах `кружок`, `класс` и `часы` заполнено меньше 15 значений — есть пропуски;
- `класс` имеет текстовый тип (`object`, а в новых версиях pandas — `str`), хотя по смыслу это число. Посмотрим на сами значения.

In [ ]:
print(df["класс"].tolist())
print(df["класс"].dtype)

`tolist()` превращает столбец в обычный список Python — так удобно увидеть все значения сразу.

Тип текстовый: из-за слова «девятый» pandas не смог прочитать столбец как числа. А пустая строка из CSV превратилась в пропуск `nan` («not a number» — «не число»).

In [ ]:
df.describe()

Максимальный балл — 200 при шкале 0–100. Это выброс.

## 4. Пропуски: сколько и где

`df.isna()` — таблица `True / False`: пусто ли в ячейке. `.sum()` по ней считает пропуски в каждом столбце (`True` считается как 1). А `.mean()` даёт **долю** пропусков: среднее из единиц и нулей — это доля единиц (например, среднее от 1, 0, 0, 0 равно 0.25 — четверть). Умножив на 100, получаем проценты.

In [ ]:
print(df.isna().sum())
print()
print("Доля пропусков по столбцам, %:")
print((df.isna().mean() * 100).round(1))

## 5. Дубликаты

- **Полный дубликат** — совпадают все столбцы.
- **Дубликат по ключу** — совпадает `id`, а остальное может различаться. Это одна и та же запись, внесённая дважды.

In [ ]:
print("Полных дубликатов:", df.duplicated().sum())
print("Дубликатов по id:", df.duplicated(subset=["id"]).sum())
df[df.duplicated(subset=["id"], keep=False)]

`subset=["id"]` — сравнивать строки только по столбцу `id`, а не по всем.

Вика (`id = 3`) записана дважды, и строки совпадают полностью. `keep=False` показывает **все** повторы, чтобы их можно было сравнить глазами. Если бы строки различались (например, разные баллы), пришлось бы решать, какая из них верная.

Решение: оставляем **первую** запись (`keep="first"`). Работаем с копией таблицы, чтобы исходные данные остались для сравнения.

In [ ]:
work = df.copy()
before = len(work)
work = work.drop_duplicates(subset=["id"], keep="first")
print("Было строк:", before, "| стало:", len(work))

## 6. Текстовые категории к единому виду

`Python`, `python `, `PYTHON` — компьютер считает это разными кружками. Методы строк из урока 6 работают и для целого столбца — через `.str`:

- `.str.strip()` — убрать пробелы по краям;
- `.str.capitalize()` — первая буква большая, остальные маленькие.

In [ ]:
print("ДО:")
print(work["кружок"].value_counts(dropna=False))

work["кружок"] = work["кружок"].str.strip().str.capitalize()

print("ПОСЛЕ:")
print(work["кружок"].value_counts(dropna=False))

`dropna=False` показывает и пропуски (`NaN`) — их тоже нужно видеть.

## 7. Типы: превращаем текст в числа

В столбце `класс` есть слово «девятый». Сначала заменим слова на цифры словарём, потом превратим столбец в числа функцией `pd.to_numeric`.

`errors="coerce"` значит: всё, что не получилось превратить в число, станет пропуском, а программа не упадёт. После этого обязательно проверяем, сколько пропусков появилось.

Метод `replace` у столбца принимает **словарь замен**: ключ — что заменить, значение — на что.

In [ ]:
work["класс"] = work["класс"].replace({"восьмой": "8", "девятый": "9"})
work["класс"] = pd.to_numeric(work["класс"], errors="coerce")
print(work["класс"].tolist())
print("Тип:", work["класс"].dtype, "| пропусков:", work["класс"].isna().sum())

## 8. Выбросы: сначала посмотреть, потом решать

Выброс — значение, которое резко выбивается из ожидаемого. Причины бывают разные:

- **опечатка** (200 вместо 20 или 100) — исправить, если знаем правду, или заменить на пропуск;
- **другая шкала** (кто-то поставил балл от 0 до 1) — привести к общей шкале;
- **редкий, но настоящий** случай — **удалять нельзя**, иначе потеряем важное.

In [ ]:
work[(work["балл"] < 0) | (work["балл"] > 100)]

Правду для Зои мы не знаем. Варианты:

1. **Обрезать** до границы шкалы: `clip(0, 100)` — балл станет 100. Но это может быть неправдой.
2. **Заменить на пропуск** и потом решать, как с пропуском.
3. **Удалить строку.**

Выберем вариант 3: подставлять придуманное значение хуже, чем честно отказаться от одной записи. Оставляем только строки, где балл в пределах шкалы — обычным фильтром из урока 10.

In [ ]:
before = len(work)
work = work[(work["балл"] >= 0) & (work["балл"] <= 100)]
print("Удалено строк с невозможным баллом:", before - len(work))

## 9. Что делать с пропусками

| Стратегия | Когда подходит | Риск |
|-----------|----------------|------|
| удалить строки (`dropna`) | пропусков мало, без этого поля строка бесполезна | теряем данные |
| заполнить **медианой** (`fillna`) | число, могут быть выбросы | сглаживаем реальность |
| заполнить самым частым значением или «Неизвестно» | категория | можем исказить частоты |
| оставить пропуск и добавить столбец-флаг «было пусто» | сам факт пропуска может что-то значить | сложнее |

**Универсальной кнопки нет.** Есть решение и запись — почему оно такое.

In [ ]:
# Часы: заполняем медианой (устойчива к выбросам), но запоминаем, где было пусто
work["часы_пропуск"] = work["часы"].isna()
hours_median = work["часы"].median()
work["часы"] = work["часы"].fillna(hours_median)
print("Часы: заполнено медианой", hours_median)

# Кружок: категория. Лучше честное «Неизвестно», чем придуманный кружок
work["кружок"] = work["кружок"].fillna("Неизвестно")

# Класс: без него запись для нашей задачи бесполезна — удаляем такие строки
before = len(work)
work = work.dropna(subset=["класс"])
print("Удалено строк без класса:", before - len(work))

print()
print(work.isna().sum())

`dropna(subset=["класс"])` удаляет строки, где пусто именно в столбце `класс`.

**Антипример.** `df.dropna()` без параметров удалит строку, где пусто **хоть что-то**. Если пропуски разбросаны по разным столбцам, так можно выбросить половину таблицы. Всегда считай, сколько строк теряешь.

## 10. Паспорт после и сохранение

`reset_index(drop=True)` заново нумерует строки с нуля — после удалений в индексе остаются «дырки».

In [ ]:
work = work.reset_index(drop=True)
print("Было:", df.shape, "→ стало:", work.shape)
work.info()

In [ ]:
work.describe()

In [ ]:
work.to_csv("clean_journal.csv", index=False)
print("Сохранено: clean_journal.csv")

## 11. Журнал решений

Через неделю ты не вспомнишь, почему удалил две строки. Журнал — обязательная часть работы инженера; он понадобится и в итоговом проекте.

| Проблема | Что сделали | Сколько строк затронуто | Почему |
|----------|-------------|------------------------:|--------|
| Вика записана дважды (`id = 3`) | удалили повтор, оставили первую запись | 1 | одна и та же ученица |
| Разное написание кружков | `strip` + `capitalize` | 3 | одно значение — одна запись |
| Пропуски в `кружок` | «Неизвестно» | 2 | не выдумываем кружок |
| «девятый» в `класс` | заменили на 9 | 1 | очевидный смысл |
| Балл 200 | удалили строку | 1 | правды не знаем, выдумывать хуже |
| Пропуски в `часы` | медиана + столбец-флаг | 2 | медиана устойчива к выбросам |
| Пустой `класс` | удалили строку | 1 | без класса запись бесполезна |

## 12. Частые ошибки и заблуждения

1. **«Чем больше удалил, тем чище».** Можно выбросить нужный сигнал вместе с мусором.
2. **«Пропуск = 0».** `fillna(0)` для часов означает «занимался 0 часов» — это ложь, и она испортит среднее.
3. **«Очистил один раз — и хватит».** Придут новые данные — снова грязные. Поэтому чистят **кодом**, который можно запустить ещё раз, а не руками в Excel.
4. **Правка без копии.** Работай с `df.copy()`, чтобы было с чем сравнить.

## 13. Практика: чистим таблицу магазина

Выполни шаги по порядку и заполни журнал решений.

In [ ]:
shop = pd.DataFrame({
    "товар": ["Яблоко", "яблоко ", "Банан", "Банан", "Груша", None, "Слива"],
    "цена": ["10", "10", "двадцать", "20", "15", "12", "1500"],
    "количество": [3, 3, 5, 5, None, 2, 4],
})
shop

In [ ]:
# Шаг 1. Паспорт: info() и число пропусков по столбцам

In [ ]:
# Шаг 2. Приведи названия товаров к единому виду (strip + capitalize), пропуск → "Неизвестно"

In [ ]:
# Шаг 3. Цена → число: "двадцать" → "20", затем pd.to_numeric(..., errors="coerce")

In [ ]:
# Шаг 4. Удали полные дубликаты (сравни число строк до и после)

In [ ]:
# Шаг 5. Цена 1500 за фрукт — выброс? Реши и обоснуй. Пропуски в количестве заполни медианой.

**Мой журнал решений:**

| Проблема | Что сделал | Почему |
|----------|------------|--------|
| | | |

<details>
<summary>Пример решения</summary>

```python
p = shop.copy()
p.info()
print(p.isna().sum())

p["товар"] = p["товар"].str.strip().str.capitalize().fillna("Неизвестно")

p["цена"] = p["цена"].replace({"двадцать": "20"})
p["цена"] = pd.to_numeric(p["цена"], errors="coerce")

before = len(p)
p = p.drop_duplicates()
print("Удалено дубликатов:", before - len(p))

# 1500 — скорее всего лишние нули (15 или 150?). Правды не знаем → удаляем строку
p = p[p["цена"] < 500]
p["количество"] = p["количество"].fillna(p["количество"].median())
p = p.reset_index(drop=True)
print(p)
```
Обрати внимание: повторы «Яблоко» и «Банан» становятся дубликатами только **после** того, как названия и цены приведены к одному виду. Поэтому дубликаты ищут после нормализации — порядок шагов важен!
</details>

## Работа с ИИ-помощником

- «Почему после `to_numeric(..., errors="coerce")` появились пропуски?»
- «Чем медиана лучше среднего для заполнения пропусков в этом столбце: …?»
- Плохо: «Почисти мою таблицу». Решения по очистке принимаешь **ты** и записываешь в журнал.

## Мини-глоссарий урока 11

| Термин | Смысл |
|--------|-------|
| **Пропуск (`NaN`, `None`)** | пустое значение |
| **Дубликат** | повторная запись того же объекта |
| **Выброс** | значение, резко выбивающееся из ожидаемого |
| **`to_numeric(errors="coerce")`** | текст → число; непонятное → пропуск |
| **`fillna` / `dropna`** | заполнить / удалить пропуски |
| **`.str`** | методы строк для целого столбца |
| **Журнал решений** | таблица: проблема → действие → почему |

## Итог урока

1. Сначала **паспорт и диагностика**, потом лечение.
2. Для каждой проблемы есть несколько стратегий; выбор зависит от смысла данных.
3. Каждое решение записываем в **журнал** — так очистку можно объяснить и повторить.

## Домашнее задание (20 минут)

Составь **свой чек-лист очистки данных** из 8–10 пунктов — своими словами, по порядку действий. Каждый пункт: что проверить и какой командой.

**Мой чек-лист очистки:**

1. …

## Самопроверка

1. Чем полный дубликат отличается от дубликата по ключу?
2. Почему нельзя всегда заполнять пропуски нулём?
3. Что делает `errors="coerce"`?
4. В каком случае выброс **нельзя** удалять?
5. Зачем нужен журнал решений?

**Следующий урок (10.12.2026):** разведка данных — ищем закономерности: группы, сравнения, связи между столбцами.